# Uncertainty-aware inverse design of a nonisothermal catalyst pellet

This commit-stamped tutorial validates and optimizes one spherical CO2-methanation pellet. It uses the published Koschany four-species kinetics and particle-scale values anchored to Zimmermann, Bremer, and Sundmacher, while labeling all transport and film data that are tutorial assumptions. The primary route is one simultaneous JAX/POUNCE NLP; an independent nested state-solve route checks it.

This is a **synthetic-calibration, activity-only, single-pellet study**. It is not an experimental fit, reactor co-design, or global-optimality claim. The full equations, units, source map, and limitations are in [`docs/src/catalyst-pellet.md`](../../docs/src/catalyst-pellet.md).

In [ ]:
from pathlib import Path
import platform, subprocess, sys, time

repo_root = Path.cwd()
if not (repo_root / 'python' / 'pounce').is_dir():
    repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root / 'python'))

import jax
import numpy as np
import scipy
import pounce
from dataclasses import replace
from pounce.examples.catalyst_pellet import (
    PelletConfig, Scenario, analytical_effectiveness, egg_shell_activity,
    fit_effective_parameters, implicit_observable_jacobian,
    refine_solution, solve_design, solve_first_order_sphere,
    solve_forward, solve_nested_design, uncertainty_scenarios,
    validate_uncertainty,
)

SOURCE_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=repo_root, text=True
).strip()
MODEL_REVISION = 'catalyst-pellet-v1'
print(f'source commit: {SOURCE_COMMIT}')
print(f'model revision: {MODEL_REVISION}')
print(f'Python {platform.python_version()} | pounce {pounce.__version__}')
print(f'numpy {np.__version__} | scipy {scipy.__version__} | jax {jax.__version__}')

## Model contract before optimization

Published anchors: 2.5 mm particle diameter, 5 bar, 0.2 CO2 / 0.8 H2, 4500 kg m^-3 solid density, 2.5 W m^-1 K^-1 conductivity, and the Koschany kinetic constants. We use 555 K (the kinetic reference temperature) rather than reproduce the source paper's axial reactor inlet. Porosity, effective diffusivities, external films, heat of reaction, catalyst inventory, and regularization are explicit assumptions. The 613 K ceiling is the top of the published kinetic range and is stricter than Zimmermann's particle-design ceiling.

In [ ]:
config = PelletConfig(nodes=8, zones=4)
print('mesh / activity zones:', config.nodes, '/', config.zones)
print('solver tolerances: forward 1e-11; design tol 1e-7; acceptable 1e-6')
print('radius [m], pressure [bar], bulk T [K]:',
      config.radius_m, config.pressure_bar, config.bulk_temperature_k)
print('D_eff [m^2/s]:', config.effective_diffusivities_m2_s)
print('mass films [m/s], heat film [W/m^2/K]:',
      config.mass_transfer_coefficients_m_s,
      config.heat_transfer_coefficient_w_m2_k)
print('inventory / upper bound / regularization:',
      config.activity_inventory, config.activity_upper,
      config.regularization_weight)

## 1. Analytical limit and conservative uniform pellet

The equal-volume sphere has a zero-area center face, so center symmetry is exact and no numerical `1/r` is evaluated. First we compare its first-order isothermal effectiveness factor with `3/phi (coth(phi) - 1/phi)`.

In [ ]:
print('phi    analytical       finite-volume    relative error    balance')
for phi in (0.0, 0.1, 1.0, 10.0):
    eta_fv, _, _, balance = solve_first_order_sphere(phi, nodes=160)
    eta_exact = analytical_effectiveness(phi)
    relative = abs(eta_fv - eta_exact) / max(abs(eta_exact), 1e-15)
    print(f'{phi:4.1f}  {eta_exact:14.10f}  {eta_fv:14.10f}  '
          f'{relative:14.3e}  {balance:9.2e}')

In [ ]:
uniform_activity = np.full(config.zones, config.activity_inventory)
uniform = solve_forward(uniform_activity, config)
uniform_refined = refine_solution(uniform, config, nodes=12)
slow_film_config = replace(
    config, mass_transfer_coefficients_m_s=(0.008,) * 4
)
film_limited = solve_forward(uniform_activity, slow_film_config)
print('case          nodes  production [mol/s]  Tmax [K]   max R     species bal  energy bal')
for name, solution in [
    ('uniform', uniform), ('refined', uniform_refined),
    ('slow film', film_limited),
]:
    print(f'{name:12s} {solution.radius_m.size:5d}  '
          f'{solution.production_mol_s:18.10e}  '
          f'{solution.max_temperature_k:8.3f}  '
          f'{solution.max_scaled_residual:8.1e}  '
          f'{np.max(solution.species_balance_relative):11.1e}  '
          f'{solution.energy_balance_relative:10.1e}')
print('coarse-to-refined production change [%]:',
      100 * (uniform_refined.production_mol_s / uniform.production_mol_s - 1))
print('coarse-to-refined Tmax change [K]:',
      uniform_refined.max_temperature_k - uniform.max_temperature_k)

## 2. Exact design derivatives and transcription choice

For fixed activity, `R(s,a)=0`. JAX differentiates the finite-volume residual and the nested route uses `ds/dq = -R_s^-1 R_q`. We compare signed production and hot-spot derivatives for every activity zone and both uncertain log parameters with full central perturb-and-resolve calculations, not with a fixture built from the derivative formula.

In [ ]:
exact = implicit_observable_jacobian(uniform, config)
epsilon = 1e-5
columns = []
for j in range(config.zones):
    plus, minus = uniform_activity.copy(), uniform_activity.copy()
    plus[j] += epsilon; minus[j] -= epsilon
    sp = solve_forward(plus, config, initial_state=uniform.state_scaled)
    sm = solve_forward(minus, config, initial_state=uniform.state_scaled)
    columns.append((
        np.array([sp.production_mol_s, sp.max_temperature_k])
        - np.array([sm.production_mol_s, sm.max_temperature_k])
    ) / (2 * epsilon))
finite_difference = np.stack(columns, axis=1)
relative_error = np.abs(exact - finite_difference) / np.maximum(
    np.abs(finite_difference), 1e-15
)
print('exact Jacobian [production; Tmax]:\n', exact)
print('full perturb-and-resolve Jacobian:\n', finite_difference)
print('maximum activity-gradient relative errors:', relative_error.max(axis=1))

scenario_exact = implicit_observable_jacobian(
    uniform, config, with_respect_to='scenario'
)
scenario_columns = []
for j in range(2):
    displacement = np.zeros(2); displacement[j] = epsilon
    sp = solve_forward(
        uniform_activity, config, scenario=Scenario(*displacement),
        initial_state=uniform.state_scaled,
    )
    sm = solve_forward(
        uniform_activity, config, scenario=Scenario(*(-displacement)),
        initial_state=uniform.state_scaled,
    )
    scenario_columns.append((
        np.array([sp.production_mol_s, sp.max_temperature_k])
        - np.array([sm.production_mol_s, sm.max_temperature_k])
    ) / (2 * epsilon))
scenario_finite_difference = np.stack(scenario_columns, axis=1)
scenario_relative_error = np.abs(
    scenario_exact - scenario_finite_difference
) / np.maximum(np.abs(scenario_finite_difference), 1e-15)
print('signed scenario Jacobian [log rate, log diffusivity]:\n',
      scenario_exact)
print('scenario full perturb-and-resolve Jacobian:\n',
      scenario_finite_difference)
print('maximum scenario-gradient relative errors:',
      scenario_relative_error.max(axis=1))

In [ ]:
started = time.perf_counter()
simultaneous = solve_design(config)
simultaneous_time = time.perf_counter() - started
started = time.perf_counter()
nested = solve_nested_design(config)
nested_time = time.perf_counter() - started
second_start = solve_design(config, initial_activity=egg_shell_activity(config))
print('route          success  iterations  wall [s]  max violation')
print(f'simultaneous  {simultaneous.success!s:7s}  {simultaneous.iterations:10d}  '
      f'{simultaneous_time:8.3f}  {simultaneous.max_constraint_violation:13.2e}')
print(f'nested        {nested.success!s:7s}  {nested.iterations:10d}  '
      f'{nested_time:8.3f}  {nested.max_constraint_violation:13.2e}')
print('simultaneous activity:', simultaneous.activity)
print('nested activity:      ', nested.activity)
print('second-start activity:', second_start.activity)
print('max simultaneous/nested activity difference:',
      np.max(np.abs(simultaneous.activity - nested.activity)))
print('max two-start activity difference:',
      np.max(np.abs(simultaneous.activity - second_start.activity)))

The routes agree, but the simultaneous transcription is the tutorial route: POUNCE sees every state bound, balance, inventory equation, and thermal limit in one NLP. The nested route is valuable as an independent algorithm and derivative check. Timings include cached JAX compilation state and are descriptive for this recorded environment, not a general benchmark.

## 3. Equal-inventory designs and independent mesh refinement

The ideal egg-shell fills the outermost volume first. The optimized design pays a fixed quadratic jump penalty, so it should be smoother and may give up some production. Classical bounded-loading work predicts step-like profiles; no global optimum is claimed.

In [ ]:
egg_activity = egg_shell_activity(config)
egg = solve_forward(egg_activity, config)
optimized_refined = refine_solution(simultaneous.nominal, config, nodes=12)
egg_refined = refine_solution(egg, config, nodes=12)
print('design       activity                              production [mol/s]  Tmax [K]  roughness')
for name, activity, solution in [
    ('uniform', uniform_activity, uniform_refined),
    ('egg-shell', egg_activity, egg_refined),
    ('optimized', simultaneous.activity, optimized_refined),
]:
    print(f'{name:11s} {np.array2string(activity, precision=4):38s} '
          f'{solution.production_mol_s:18.10e}  '
          f'{solution.max_temperature_k:8.3f}  '
          f'{np.sum(np.diff(activity)**2):9.4f}')
print('optimized refinement changes: production [%], Tmax [K] =',
      100 * (optimized_refined.production_mol_s
             / simultaneous.nominal.production_mol_s - 1),
      optimized_refined.max_temperature_k
      - simultaneous.nominal.max_temperature_k)

Zimmermann et al. found an active **core** with an inert low-permeability shell while also optimizing permeability, conductivity, and reactor operation. This activity-only prescribed-bulk model instead favors outer activity. The near-step structure agrees qualitatively with bounded-loading theory, but the opposite placement is an important model-form difference, not a claimed reproduction of the coupled reactor optimum.

## 4. Synthetic calibration, covariance, and uncertainty propagation

Synthetic log-rate data combine intrinsic powder observations with apparent rates at two pellet radii. POUNCE fits log multipliers for intrinsic rate and CO2 diffusivity and obtains covariance from its reduced Hessian. Principal-axis scenarios use 1.645 standard deviations. Delta-method uncertainty is checked against full sampled nonlinear re-solves.

In [ ]:
fit = fit_effective_parameters(config=config)
scenarios = uncertainty_scenarios(fit.popt, fit.pcov)
validation = validate_uncertainty(
    simultaneous.activity, fit.popt, fit.pcov, config, samples=16, seed=103
)
print('fit success / covariance route:', fit.success, '/', fit.cov_source)
print('parameters [log rate, log diffusivity]:', fit.popt)
print('standard errors:', fit.perr)
print('correlation:\n', fit.correlation)
print('observable          nominal          delta sd        sampled sd      ratio')
for i, name in enumerate(validation.observable_names):
    print(f'{name:20s} {validation.nominal[i]:14.6e}  '
          f'{validation.delta_standard_deviation[i]:14.6e}  '
          f'{validation.sampled_standard_deviation[i]:14.6e}  '
          f'{validation.sampled_standard_deviation[i] / validation.delta_standard_deviation[i]:7.3f}')
print('sampled Tmax range [K]:',
      validation.sampled_minimum[1], validation.sampled_maximum[1])

## 5. Scenario-based worst-case redesign

All covariance scenarios share one activity profile. An epigraph variable is constrained below every scenario production, and POUNCE maximizes that finite-set guarantee minus the same manufacturability penalty. It is not a distribution-free guarantee.

In [ ]:
fitted_nominal = solve_design(config, scenarios=(scenarios[0],))
robust = solve_design(
    config, scenarios=scenarios, robust=True,
    initial_activity=fitted_nominal.activity,
)
nominal_scenario_solutions = [
    solve_forward(fitted_nominal.activity, config, scenario=scenario)
    for scenario in scenarios
]
print('nominal activity:', fitted_nominal.activity)
print('robust activity: ', robust.activity)
print('scenario             nominal design [mol/s]  robust design [mol/s]  robust Tmax [K]')
for scenario, nominal_solution, robust_solution in zip(
    scenarios, nominal_scenario_solutions, robust.solutions
):
    print(f'{scenario.label:20s} {nominal_solution.production_mol_s:22.10e}  '
          f'{robust_solution.production_mol_s:21.10e}  '
          f'{robust_solution.max_temperature_k:14.3f}')
scenario_reference = np.mean([
    solve_forward(uniform_activity, config, scenario=scenario).production_mol_s
    for scenario in scenarios
])
nominal_score = (
    min(solution.production_mol_s for solution in nominal_scenario_solutions)
    / scenario_reference
    - config.regularization_weight * np.sum(np.diff(fitted_nominal.activity)**2)
)
robust_score = (
    min(solution.production_mol_s for solution in robust.solutions)
    / scenario_reference
    - config.regularization_weight * np.sum(np.diff(robust.activity)**2)
)
print('enforced finite-scenario production floor [mol/s]:',
      robust.guaranteed_production_mol_s)
print('observed minimum over scenario re-solves [mol/s]:',
      min(solution.production_mol_s for solution in robust.solutions))
print('worst-case regularized score, nominal / robust:',
      nominal_score, '/', robust_score)
print('maximum robust-scenario temperature [K]:',
      max(solution.max_temperature_k for solution in robust.solutions),
      '<', config.temperature_limit_k)

## What this result does and does not establish

The analytical limit, integrated balances, film response, mesh refinement, full re-solve gradient check, independent nested optimizer, two starts, covariance construction, and sampled uncertainty re-solves all pass in the recorded run. The recommended profile is still only a **local optimum of this model and this activity basis**. Assumed transport data and omitted reactor/pore physics are model-form uncertainty. The covariance is local and synthetic; the principal-axis set is finite. No claim is made about a globally optimal pellet architecture, experimental robustness, permeability or conductivity design, transient operation, or a dead core.